# 01 Data Preparation

Project: Cat Breed Classification Using CNN Transfer Learning  
Domain: Animal subspecies  
Dataset: Oxford-IIIT Pet Dataset

This notebook prepares a filtered cat breed dataset for the AI project. It will:

1. Download the Oxford-IIIT Pet images and annotations.
2. Filter the dataset to five selected cat breeds.
3. Create training, validation, and testing folders.
4. Visualize class distribution.
5. Show example images from each class.

Selected classes:

- Abyssinian
- Bengal
- Birman
- Bombay
- British Shorthair

## 1. Setup

Run this notebook in Google Colab or a local Jupyter environment. If you use Google Colab, the dataset will be created inside the current Colab working directory unless you mount Google Drive.

In [ ]:
import os
import tarfile
import random
import shutil
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

random.seed(42)

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "raw_data"
DATASET_DIR = PROJECT_ROOT / "dataset"

RAW_DIR.mkdir(exist_ok=True)
DATASET_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Prepared dataset folder:", DATASET_DIR)

Optional for Google Colab: uncomment the cell below if you want to save the prepared dataset directly into Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = Path('/content/drive/MyDrive/AI-Image-Classification-Project')
# RAW_DIR = PROJECT_ROOT / 'raw_data'
# DATASET_DIR = PROJECT_ROOT / 'dataset'
# RAW_DIR.mkdir(parents=True, exist_ok=True)
# DATASET_DIR.mkdir(parents=True, exist_ok=True)

## 2. Download Oxford-IIIT Pet Dataset

Official dataset page: https://www.robots.ox.ac.uk/~vgg/data/pets/

The images and annotations are downloaded from the official Oxford VGG source.

In [ ]:
IMAGES_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
ANNOTATIONS_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"

images_archive = RAW_DIR / "images.tar.gz"
annotations_archive = RAW_DIR / "annotations.tar.gz"

!wget -nc -O "{images_archive}" "{IMAGES_URL}"
!wget -nc -O "{annotations_archive}" "{ANNOTATIONS_URL}"

In [ ]:
def extract_tar_gz(archive_path, destination):
    marker = destination / (archive_path.stem + ".extracted")
    if marker.exists():
        print(f"Already extracted: {archive_path.name}")
        return
    print(f"Extracting {archive_path.name}...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(destination)
    marker.touch()
    print("Done")

extract_tar_gz(images_archive, RAW_DIR)
extract_tar_gz(annotations_archive, RAW_DIR)

images_dir = RAW_DIR / "images"
annotations_dir = RAW_DIR / "annotations"

print("Images folder exists:", images_dir.exists())
print("Annotations folder exists:", annotations_dir.exists())

## 3. Select Cat Breed Classes

The Oxford-IIIT Pet image filenames contain the breed name followed by an image number, for example `Abyssinian_1.jpg`.

In [ ]:
SELECTED_CLASSES = {
    "Abyssinian": "Abyssinian",
    "Bengal": "Bengal",
    "Birman": "Birman",
    "Bombay": "Bombay",
    "British_Shorthair": "British_Shorthair",
}

def get_breed_name(image_path):
    stem = image_path.stem
    return "_".join(stem.split("_")[:-1])

all_images = sorted(images_dir.glob("*.jpg"))
selected_images = [p for p in all_images if get_breed_name(p) in SELECTED_CLASSES]

print("Total images in full dataset:", len(all_images))
print("Selected images:", len(selected_images))
print("Selected class counts:")
Counter(get_breed_name(p) for p in selected_images)

## 4. Check For Corrupted Images

This step verifies that images can be opened by PIL. Any corrupted files are skipped.

In [ ]:
valid_images = []
corrupted_images = []

for image_path in selected_images:
    try:
        with Image.open(image_path) as img:
            img.verify()
        valid_images.append(image_path)
    except Exception as error:
        corrupted_images.append((image_path, str(error)))

print("Valid selected images:", len(valid_images))
print("Corrupted selected images:", len(corrupted_images))

if corrupted_images:
    for image_path, error in corrupted_images[:10]:
        print(image_path.name, error)

## 5. Create Train, Validation, And Test Splits

Recommended split:

- 70% training
- 15% validation
- 15% testing

The split is stratified so every class appears in each folder.

In [ ]:
records = []
for image_path in valid_images:
    breed = get_breed_name(image_path)
    records.append({"path": image_path, "class_name": SELECTED_CLASSES[breed]})

df = pd.DataFrame(records)
df.head()

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["class_name"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["class_name"],
    random_state=42,
)

split_frames = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

for split_name, split_df in split_frames.items():
    print(split_name, len(split_df))
    print(split_df["class_name"].value_counts().sort_index())
    print()

## 6. Copy Images Into Folder Structure

This creates the format expected by Keras image loading utilities:

```text
dataset/
  train/
    Abyssinian/
    Bengal/
    Birman/
    Bombay/
    British_Shorthair/
  validation/
    ...
  test/
    ...
```

In [ ]:
def prepare_split_folder(split_name, split_df):
    split_dir = DATASET_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    for class_name in SELECTED_CLASSES.values():
        (split_dir / class_name).mkdir(parents=True, exist_ok=True)

    for _, row in split_df.iterrows():
        source_path = row["path"]
        class_name = row["class_name"]
        destination_path = split_dir / class_name / source_path.name
        shutil.copy2(source_path, destination_path)

for split_name, split_df in split_frames.items():
    prepare_split_folder(split_name, split_df)

print("Dataset created at:", DATASET_DIR)

## 7. Save Split Metadata

The CSV files make the split reproducible and easier to explain in the report/presentation.

In [ ]:
metadata_dir = PROJECT_ROOT / "results"
metadata_dir.mkdir(exist_ok=True)

for split_name, split_df in split_frames.items():
    export_df = split_df.copy()
    export_df["path"] = export_df["path"].astype(str)
    export_df.to_csv(metadata_dir / f"{split_name}_files.csv", index=False)

print("Saved split metadata CSV files in:", metadata_dir)

## 8. Visualize Class Distribution

In [ ]:
distribution_rows = []

for split_name, split_df in split_frames.items():
    counts = split_df["class_name"].value_counts().sort_index()
    for class_name, count in counts.items():
        distribution_rows.append({
            "split": split_name,
            "class_name": class_name,
            "count": count,
        })

distribution_df = pd.DataFrame(distribution_rows)
distribution_pivot = distribution_df.pivot(index="class_name", columns="split", values="count")
distribution_pivot = distribution_pivot[["train", "validation", "test"]]
distribution_pivot

In [ ]:
ax = distribution_pivot.plot(kind="bar", figsize=(10, 5))
ax.set_title("Cat Breed Dataset Split Distribution")
ax.set_xlabel("Cat Breed")
ax.set_ylabel("Number of Images")
ax.legend(title="Split")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

graphs_dir = PROJECT_ROOT / "results" / "graphs"
graphs_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(graphs_dir / "class_distribution.png", dpi=150)
plt.show()

## 9. Show Example Images

In [ ]:
fig, axes = plt.subplots(len(SELECTED_CLASSES), 5, figsize=(12, 10))

for row_idx, class_name in enumerate(SELECTED_CLASSES.values()):
    class_images = list((DATASET_DIR / "train" / class_name).glob("*.jpg"))
    sample_images = random.sample(class_images, min(5, len(class_images)))
    for col_idx, image_path in enumerate(sample_images):
        image = Image.open(image_path).convert("RGB")
        axes[row_idx, col_idx].imshow(image)
        axes[row_idx, col_idx].axis("off")
        if col_idx == 0:
            axes[row_idx, col_idx].set_title(class_name, loc="left")

plt.tight_layout()
plt.savefig(graphs_dir / "sample_cat_breeds.png", dpi=150)
plt.show()

## 10. Final Dataset Summary

In [ ]:
summary = {
    "domain": "Animal subspecies",
    "dataset": "Oxford-IIIT Pet Dataset",
    "number_of_classes": len(SELECTED_CLASSES),
    "classes": list(SELECTED_CLASSES.values()),
    "train_images": len(train_df),
    "validation_images": len(val_df),
    "test_images": len(test_df),
    "total_images": len(df),
}

summary

## Next Step

After this notebook is complete, continue with:

- `02_resnet50_training.ipynb`
- `03_densenet121_training.ipynb`
- `04_mobilenetv3_training.ipynb`
- `05_model_comparison.ipynb`